# Arbitrary Interval Table

In [1]:
# Import packages
import functools
import polars as pl
import hicona as hi

# Data paths 
COOLER_PATH = "../data/IMR90_100kb_chr17_balanced.cool"
handle = hi.HiconaCooler(COOLER_PATH)

Currently, genomic intervals must always be specified in the form `chrom:start-end`. This means that it is not possible to directly create a subset table with inter-chromosomal pixels or discontinuous intervals on the same chromosome (e.i. skip the centromere). While this behavior will likely change in the future, currently there are two possible workarounds.

## Merging individual tables

If the table of interest can be represented by the union of tables generated by specifying genomic intervals in `chrom:start-end` format, the easiest way is to create a new table where pixels are provided via an iterable constructed from the sub tables.

In [2]:
table_a = handle.get_pixel_table("chr17:0-1000000")
table_b = handle.get_pixel_table("chr17:4000000-8000000")
table_c = hi.PixelTable(bins=table_a.bins, pixels=(table_a.get_dataframe(), table_b.get_dataframe()))

print(len(table_a.get_dataframe()) + len(table_b.get_dataframe()) == len(table_c.get_dataframe()))

True


This approach has some limitations:

- no check is performed that the two tables use the same binning. It is your responsibility to perform this check.
- it still does not allow to keep inter chromosomal pixels.
- it requires the whole sub pixel tables to be loaded into memory at once. In the rare eventuality that the sub tables do not fit in memory, you might need to create a custom iterable which iterates the `get_chunks` method of each table until exhausted, while returning those chunks one at a time to the `pixels` argument.

## Custom filtering

As already mentioned in the `HiconaCooler` and `PixelTable` notebooks, there is the possibility to define a custom filtering function to pass either to the strategy argument of `PixelTable.get_chunks` and `PixelTable.get_dataframe` or to the `PixelTable.apply` method. This means that we can start from the full `PixelTable` and apply a custom genomic filter to create a table of interest.

In [3]:
def filter_inter_regions(chunks, region1, region2):
    "Return only pixels with one bin per provided region."
    
    pix1_in_chr_1 = (pl.col("bin1_id") >= region1[0]) & (pl.col("bin1_id") < region1[1])
    pix1_in_chr_2 = (pl.col("bin1_id") >= region2[0]) & (pl.col("bin1_id") < region2[1])
    pix2_in_chr_1 = (pl.col("bin2_id") >= region1[0]) & (pl.col("bin2_id") < region1[1])
    pix2_in_chr_2 = (pl.col("bin2_id") >= region2[0]) & (pl.col("bin2_id") < region2[1])
    extent_filter = (pix1_in_chr_1 & pix2_in_chr_2) | (pix2_in_chr_1 & pix1_in_chr_2)

    for chunk in chunks:
        yield chunk.filter(extent_filter)

full_table = handle.get_pixel_table()

region1 = full_table.bins.extent("chr17:0-1000000")
region2 = full_table.bins.extent("chr17:4000000-8000000")  # Can use genomic regions on different chromosomes
custom_filter = functools.partial(filter_inter_regions, region1=region1, region2=region2)

subset_table = full_table.apply(custom_filter)
print(subset_table.get_dataframe())

shape: (363, 3)
┌─────────┬─────────┬───────┐
│ bin1_id ┆ bin2_id ┆ count │
│ ---     ┆ ---     ┆ ---   │
│ i64     ┆ i64     ┆ i32   │
╞═════════╪═════════╪═══════╡
│ 26475   ┆ 26518   ┆ 1     │
│ 26475   ┆ 26526   ┆ 1     │
│ 26475   ┆ 26534   ┆ 1     │
│ 26476   ┆ 26515   ┆ 15    │
│ 26476   ┆ 26516   ┆ 7     │
│ …       ┆ …       ┆ …     │
│ 26484   ┆ 26550   ┆ 29    │
│ 26484   ┆ 26551   ┆ 22    │
│ 26484   ┆ 26552   ┆ 19    │
│ 26484   ┆ 26553   ┆ 29    │
│ 26484   ┆ 26554   ┆ 20    │
└─────────┴─────────┴───────┘


This approach allows to use any custom interval, though it is more difficult to use. By default a strategy takes as input an iterable of `polars.DataFrame` objects; if you are not familiar with this API, simply convert each dataframe to `pandas` by using `df.to_pandas()`.